# OpenVLA-OFT resident adapter (`--backend openvla-oft`)

Serves OpenVLA-OFT from its own resident Flask adapter, `control/vla_bridge/openvla_oft_server.py`, through an ngrok HTTP tunnel.

| | |
|---|---|
| Lane | HTTP adapter (isolated Python 3.10 venv) |
| Port → tunnel | 3001 → ngrok HTTP |
| GPU runtime | L4 |
| Gated HF licences | none |
| Sign-offs before running | 14-UAT 4 package-legitimacy sign-off first |
| Default checkpoint | `moojink/openvla-7b-oft-finetuned-libero-object@4c89574e1c53` |
| Server log | `/content/openvla_oft_server.log` |
| Embedded files (written to `/content`) | `colab_server_utils.py`, `openvla_oft_server.py` |

Run top to bottom on a fresh runtime. Works from the VS Code Colab extension: nothing to upload.
Every file this notebook runs is written into `/content` by a cell below and checked against
its sha256. Run the teardown cell at the end of every session. The index of all notebooks,
with the local smoke and episode commands, is `control/vla_bridge/notebooks/README.md`.

Generated by `scripts/build_colab_notebooks.py` from `scripts/colab_notebook_specs.py` +
`control/vla_bridge` sources, so edit those and regenerate, not this file.

In [ ]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

**pyngrok legitimacy gate:** the next cell installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

In [ ]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

In [ ]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

## Server helper

The next cell writes `control/vla_bridge/colab_server_utils.py` into `/content/colab_server_utils.py`; the cell after it checks its sha256. The launch, diagnostic and teardown cells import it to stop stale servers, prove this notebook's server owns its port, and report logs, processes and GPU memory.

In [ ]:
%%writefile /content/colab_server_utils.py
"""Colab-side process/port helper for the per-backend policy-server notebooks.

Every notebook in control/vla_bridge/notebooks/ writes this file to
/content/colab_server_utils.py with a %%writefile cell (byte-identical to this file, checked
by a sha256 cell and by control/test_colab_notebooks.py) and imports it with
sys.path.insert(0, '/content'). Local control code never imports it.

What it guarantees for a launch cell:
- stop_servers() clears any earlier server: every process whose cmdline contains a known
  server marker, plus whatever is listening on the notebook's port.
- start_server() starts the server unbuffered (PYTHONUNBUFFERED=1) with its log in a file.
- wait_until_ready() only succeeds when the process this cell started (or one of its
  children) owns the port. A stale server that answers on the port is an error, not a
  success. It also fails on an early exit and, for patched launchers, on a missing banner.

The diagnostic helpers (log_report, process_report, gpu_report, health_report) print what a
partner needs to debug a run, and stop_server() is the teardown.

Imports: stdlib + psutil only (psutil is preinstalled on Colab).
"""

import json
import os
import re
import socket
import subprocess
import time
import urllib.request
from pathlib import Path
from typing import Callable, Mapping, Optional, Sequence

import psutil

SERVER_MARKERS = (
    "lerobot.async_inference.policy_server",
    "diffusion_policy_server.py",
    "groot_n17_server.py",
    "fastfs_server.py",
    "openvla_oft_server.py",
    "openvla_server.py",
    "octo_server.py",
    "wallx_server.py",
    "rdt1b_server.py",
    "groot_n15_server.py",
)

# Case-sensitive on purpose: "Error" covers KeyError, NotImplementedError, GatedRepoError and
# "Error in StreamActions"; lowercase prose such as "error budget" is not a match.
ERROR_PATTERN = re.compile(
    r"ERROR|Error|Traceback|Exception|OutOfMemory|out of memory|Killed|beginAllocateToPool"
)

_ACCESS_ERRORS = (psutil.AccessDenied, psutil.NoSuchProcess, psutil.ZombieProcess)


def _cmdline(proc) -> str:
    try:
        return " ".join(proc.cmdline())
    except _ACCESS_ERRORS:
        return ""
    except OSError:
        return ""


def _describe(pid: int) -> str:
    try:
        return f"PID {pid} ({_cmdline(psutil.Process(pid)) or '?'})"
    except _ACCESS_ERRORS:
        return f"PID {pid} (gone)"


def _tcp_connections(proc):
    # Colab's psutil may predate 6.0, where net_connections() replaced connections().
    method = getattr(proc, "net_connections", None) or proc.connections
    return method(kind="tcp")


def _read_lines(log_path) -> list:
    path = Path(log_path)
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()


def _tail(log_path, lines: int = 40) -> str:
    return "\n".join(_read_lines(log_path)[-lines:])


def listening_pids(port: int) -> set:
    """PIDs of every visible process with a TCP socket in LISTEN state on `port`."""
    pids = set()
    for proc in psutil.process_iter():
        try:
            for conn in _tcp_connections(proc):
                if conn.status == psutil.CONN_LISTEN and conn.laddr and conn.laddr.port == port:
                    pids.add(proc.pid)
        except _ACCESS_ERRORS:
            continue
        except OSError:
            continue
    return pids


def stop_servers(port: int, patterns: Sequence[str] = (), timeout_s: float = 15.0) -> list:
    """Stop every process matching a pattern plus the port's listener; return what was stopped."""
    me = os.getpid()
    targets = {}
    for proc in psutil.process_iter():
        if proc.pid == me:
            continue
        cmdline = _cmdline(proc)
        if cmdline and any(pattern and pattern in cmdline for pattern in patterns):
            targets[proc.pid] = proc
    for pid in listening_pids(port):
        if pid != me and pid not in targets:
            try:
                targets[pid] = psutil.Process(pid)
            except _ACCESS_ERRORS:
                continue
    stopped = []
    for pid, proc in sorted(targets.items()):
        description = f"PID {pid} ({_cmdline(proc) or '?'})"
        try:
            proc.terminate()
            stopped.append(description)
        except psutil.NoSuchProcess:
            continue
        except psutil.AccessDenied:
            print(f"Cannot stop {description}: access denied")
    _, alive = psutil.wait_procs(list(targets.values()), timeout=5)
    for proc in alive:
        try:
            proc.kill()
        except psutil.NoSuchProcess:
            pass
    psutil.wait_procs(alive, timeout=5)
    for line in stopped:
        print(f"Stopped {line}")
    deadline = time.monotonic() + timeout_s
    while True:
        holders = listening_pids(port) - {me}
        if not holders:
            return stopped
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"port {port} is still held by "
                + ", ".join(_describe(pid) for pid in sorted(holders))
                + f" after {timeout_s:.0f} s; stop it by hand before launching"
            )
        time.sleep(0.2)


def start_server(argv: Sequence[str], log_path,
                 env_overrides: Optional[Mapping[str, str]] = None,
                 cwd=None) -> subprocess.Popen:
    """Start `argv` unbuffered with stdout+stderr in `log_path` (truncated)."""
    env = os.environ.copy()
    env.update(env_overrides or {})
    env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "wb", buffering=0)
    try:
        proc = subprocess.Popen(list(argv), stdout=log, stderr=subprocess.STDOUT, env=env,
                                cwd=None if cwd is None else str(cwd))
    finally:
        log.close()
    print(f"Started PID {proc.pid}: {' '.join(argv)}")
    print(f"Log → {log_path}")
    return proc


def _probe(port: int, health_url: Optional[str], ready: Optional[Callable[[dict], bool]]):
    """Return (answered, health): answered is True once the server counts as ready."""
    if health_url:
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                health = json.loads(response.read().decode())
        except Exception:
            return False, None
        if not isinstance(health, dict):
            return False, None
        ok = ready(health) if ready is not None else health.get("status") == "ready"
        return bool(ok), health
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True, None
    except OSError:
        return False, None


def _own_pids(proc) -> set:
    pids = {proc.pid}
    try:
        pids.update(child.pid for child in psutil.Process(proc.pid).children(recursive=True))
    except _ACCESS_ERRORS:
        pass
    return pids


def wait_until_ready(proc, port: int, log_path, timeout_s: float,
                     health_url: Optional[str] = None,
                     ready: Optional[Callable[[dict], bool]] = None,
                     banner: Optional[str] = None,
                     settle_s: float = 2.0, poll_s: float = 1.0):
    """Wait until the server this cell started owns `port` and is ready.

    Returns the /health dict in health mode, None in TCP mode. Raises RuntimeError on an
    early exit, a foreign/stale owner or a missing launcher banner, and TimeoutError (after
    terminating `proc`) when the deadline passes.
    """
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        code = proc.poll()
        if code is not None:
            raise RuntimeError(
                f"server PID {proc.pid} exited with code {code} before it was ready. "
                f"Last log lines of {log_path}:\n{_tail(log_path)}"
            )
        answered, health = _probe(port, health_url, ready)
        if answered:
            owners = listening_pids(port)
            mine = _own_pids(proc)
            if not owners or not owners <= mine:
                foreign = ", ".join(_describe(pid) for pid in sorted(owners - mine))
                foreign = foreign or "no visible process"
                raise RuntimeError(
                    f"port {port} answered but is owned by {foreign}, not the server this cell "
                    f"started (PID {proc.pid}). That is a stale server; re-run the launch cell, "
                    "which stops old servers first."
                )
            time.sleep(settle_s)
            code = proc.poll()
            if code is not None:
                raise RuntimeError(
                    f"server PID {proc.pid} exited with code {code} right after binding port "
                    f"{port}. Last log lines of {log_path}:\n{_tail(log_path)}"
                )
            if banner is not None:
                text = Path(log_path).read_text(errors="replace") if Path(log_path).exists() else ""
                if banner not in text:
                    raise RuntimeError(
                        f"expected launcher banner '{banner}' not found in {log_path}, so this "
                        "is not the patched launcher. Re-run the embed cells, then the launch cell."
                    )
            print(f"READY: PID {proc.pid} ({_cmdline(psutil.Process(proc.pid))}) owns port "
                  f"{port}; log → {log_path}")
            return health
        time.sleep(poll_s)
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
    raise TimeoutError(
        f"server PID {proc.pid} was not ready on port {port} within {timeout_s:.0f} s "
        f"(terminated). Last log lines of {log_path}:\n{_tail(log_path)}"
    )


def log_report(log_path, tail: int = 80, max_matches: int = 40) -> None:
    """Print the last `tail` log lines and up to `max_matches` numbered error lines."""
    path = Path(log_path)
    if not path.exists():
        print(f"{path} does not exist (has the launch cell run on this runtime?)")
        return
    lines = _read_lines(path)
    print(f"--- last {tail} lines of {path} ({len(lines)} lines total) ---")
    for line in lines[-tail:]:
        print(line)
    matches = [(number, line) for number, line in enumerate(lines, start=1)
               if ERROR_PATTERN.search(line)]
    print(f"--- {len(matches)} lines matching ERROR_PATTERN (showing up to {max_matches}) ---")
    for number, line in matches[:max_matches]:
        print(f"{number}: {line}")
    if not matches:
        print("(none)")


def process_report(port: int, expected_marker: str, proc=None) -> None:
    """Print every known server process, then check the right server owns `port`."""
    print("--- running server processes ---")
    found = False
    now = time.time()
    for candidate in psutil.process_iter():
        cmdline = _cmdline(candidate)
        if not cmdline or not any(marker in cmdline for marker in SERVER_MARKERS):
            continue
        try:
            elapsed = now - candidate.create_time()
            rss_mb = candidate.memory_info().rss / 1e6
        except _ACCESS_ERRORS:
            continue
        found = True
        print(f"PID {candidate.pid}  up {elapsed:7.0f} s  RSS {rss_mb:8.1f} MB  {cmdline}")
    if not found:
        print("(no process with a known server marker)")
    owners = listening_pids(port)
    if not owners:
        raise RuntimeError(f"nothing is listening on port {port}; run the launch cell")
    for pid in sorted(owners):
        try:
            cmdline = _cmdline(psutil.Process(pid))
        except _ACCESS_ERRORS:
            cmdline = ""
        if expected_marker not in cmdline:
            raise RuntimeError(
                f"WRONG server on port {port}: PID {pid} ({cmdline or '?'}) does not contain "
                f"'{expected_marker}'. Run the teardown cell, then the launch cell."
            )
    if proc is not None:
        mine = _own_pids(proc)
        if not owners <= mine:
            raise RuntimeError(
                f"port {port} is owned by "
                + ", ".join(_describe(pid) for pid in sorted(owners))
                + f", not by the server this notebook started (PID {proc.pid})"
            )
    print(f"OK: {expected_marker} (PID {min(owners)}) owns port {port}")


def gpu_report() -> None:
    """Print GPU memory/utilisation and the processes using the GPU."""
    queries = (
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv"],
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    )
    for argv in queries:
        try:
            result = subprocess.run(argv, capture_output=True, text=True, timeout=30)
        except FileNotFoundError:
            print("no nvidia-smi (CPU runtime?): pick a GPU runtime for this notebook")
            return
        print(result.stdout.rstrip() or result.stderr.rstrip())


def health_report(port: int) -> dict:
    """GET /health on the local server, pretty-print it and return it."""
    url = f"http://127.0.0.1:{port}/health"
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            health = json.loads(response.read().decode())
    except Exception as exc:
        print(f"GET {url} failed: {type(exc).__name__}: {exc}")
        raise
    print(json.dumps(health, indent=2, sort_keys=True))
    return health


def stop_server(proc, port: int, patterns: Sequence[str] = ()) -> None:
    """Teardown: stop `proc` (if alive) and any leftover server, then confirm the port is free."""
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
        print(f"Stopped PID {proc.pid}")
    stop_servers(port, patterns)
    print(f"port {port} is free")


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '2d8f63cbfdf5e679c268dd4f4436129e8fa8f30acfd5c5f5d88eb1082c2bf2a8'
actual = hashlib.sha256(Path('/content/colab_server_utils.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/colab_server_utils.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/colab_server_utils.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/colab_server_utils.py matches control/vla_bridge/colab_server_utils.py (sha256 {actual[:12]})')

## Tier C: OpenVLA-OFT resident adapter (Plan 14-03, `--backend openvla-oft`)

OpenVLA-OFT has no LeRobot `PolicyServer` integration. It runs as its own
resident Flask adapter, `control/vla_bridge/openvla_oft_server.py`, built the
same way as the resident depth service (`fastfs_depth.ipynb`): it loads once, warms up, then serves.
It does not use the gRPC PolicyServer lane.

**Status:** these cells were written from upstream source (github.com/moojink/openvla-oft
@ `e4287e9`, `SETUP.md` + `vla-scripts/deploy.py`) and were verified live on Colab
(L4) on 2026-10-25 (backend smoke PASS). The package-legitimacy human check in
14-03-SUMMARY.md must pass before the clone and install cell runs. `pip install -e .`
pulls two more GitHub-source forks, `moojink/transformers-openvla-oft` and
`moojink/dlimp_openvla`.

openvla-oft leaves `protobuf`, `wandb`, `tensorflow_metadata` and `numpy` unpinned, and
their current releases no longer import under `tensorflow==2.15.0` (numpy 2 breaks
`ml_dtypes`; tensorflow-metadata >1.14 needs protobuf >=5.26 `runtime_version`; wandb
>=0.28 needs protobuf >=5). The install cell pins the 2023-era set that works:
`numpy<2`, `protobuf==3.20.3`, `tensorflow-metadata==1.14.0`, `wandb==0.16.6`, and
`opencv-python-headless==4.11.0.86` (the last opencv that accepts numpy 1.x).

Requirements: an A100 or L4 runtime (a 7B model in bf16 needs about 16 GB of VRAM).
openvla-oft pins `torch==2.2.0` and `tensorflow==2.15.0`, so it needs Python 3.10
in its own isolated environment, never the PolicyServer's.

In [ ]:
import subprocess, sys
from pathlib import Path
OFT_DIR = Path('/content/openvla-oft')
OFT_COMMIT = 'e4287e94541f459edc4feabc4e181f537cd569a8'  # reviewed revision
OFT_VENV = '/content/openvla_oft_venv'
OFT_PYTHON = f'{OFT_VENV}/bin/python'
if not (OFT_DIR / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/moojink/openvla-oft.git', str(OFT_DIR)], check=True)
subprocess.run(['git', '-C', str(OFT_DIR), 'checkout', OFT_COMMIT], check=True)
if not Path(OFT_PYTHON).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
    subprocess.run(['uv', 'venv', '--python', '3.10', '--seed', OFT_VENV], check=True)
# Pins keep tensorflow==2.15.0 importable (numpy 1.x, protobuf 3.x); see the note above.
OFT_PINS = ['numpy<2', 'protobuf==3.20.3', 'tensorflow-metadata==1.14.0', 'wandb==0.16.6']
subprocess.run(['uv', 'pip', 'install', '--python', OFT_PYTHON, '-e', str(OFT_DIR), *OFT_PINS], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', OFT_PYTHON, 'flask',
                'opencv-python-headless==4.11.0.86', *OFT_PINS], check=True)
subprocess.run([OFT_PYTHON, '-c', 'import experiments.robot.openvla_utils, numpy, google.protobuf, wandb; '
                'print("IMPORT OK numpy", numpy.__version__, "protobuf", google.protobuf.__version__, '
                '"wandb", wandb.__version__)'], cwd=str(OFT_DIR), check=True)

## Adapter source and launch notes

The next cell writes `control/vla_bridge/openvla_oft_server.py` into `/content/openvla_oft_server.py`, and the cell after it checks its sha256. The launch cell then starts it as a separate OS process.

On first start
it downloads the pinned checkpoint revision
(`moojink/openvla-7b-oft-finetuned-libero-object@4c89574e1c53`, about 15 GB). `/health`
only answers after the model has loaded and warmed up. Logs go to
`/content/openvla_oft_server.log`.

In [ ]:
%%writefile /content/openvla_oft_server.py
"""Resident OpenVLA-OFT HTTP adapter (Tier C, Plan 14-03). Run on Colab with the
openvla-oft environment's Python interpreter.

This mirrors `fastfs_server.py`:
- the model loads ONCE
- one throwaway warmup inference runs before the socket opens
- `/health` reports ready
- a non-blocking `inference_lock` answers concurrent requests with 429
- `ValueError`/`KeyError`/`TypeError` on a request become 400

The server stays model-format-only. It returns OpenVLA-OFT's raw 7-D action,
and every `action_contract` convention is applied control-side by
`backends/openvla_oft.py` through `backends/eef_to_joint.py`.

Verified against live upstream source (2026-10-01, execution discipline)
------------------------------------------------------------------------
- Code: github.com/moojink/openvla-oft @ e4287e9 (the model card's
  "github.com/openvla-oft/openvla-oft" link returns 404). Loading follows its
  own server, `vla-scripts/deploy.py`, through
  `experiments/robot/openvla_utils.py`:
  - `get_vla(cfg)` calls `AutoModelForVision2Seq.from_pretrained(...,
    torch_dtype=bfloat16, trust_remote_code=True)`.
  - `get_proprio_projector` and `get_action_head` load the separate
    `proprio_projector--*_checkpoint.pt` / `action_head--*_checkpoint.pt`
    MLPs, which a bare `from_pretrained()` does NOT load.
  - `get_processor(cfg)` and `get_vla_action(...)` run inference.
- Config: copied from the HF model card quick start and LIBERO
  `GenerateConfig`: L1-regression head, no FiLM, 2 input images
  (third-person `full_image` + `wrist_image`), proprio on, center crop on,
  `lora_rank=32`.
- Checkpoint: moojink/openvla-7b-oft-finetuned-libero-object. HF API sha
  4c89574e1c538b6c102f43f0526d60a9d3650148, read 2026-10-01.
  `dataset_statistics.json` has the single key `libero_object_no_noops`.
  - Action: 7-D, BOUNDS_Q99-normalized, chunk of `NUM_ACTIONS_CHUNK=8`.
    After un-normalization it is a robosuite OSC_POSE delta in the
    controller's normalized [-1, 1] units (dataset min/max reach +/-0.9375).
    The gripper is in [0, 1] with 1 = open (`run_libero_eval.process_action`
    flips it for the env).
  - Proprio: 8-D LIBERO Franka state (eef pos 3 + axis-angle 3 + gripper
    qpos 2).

Known limitations (documented, not hidden)
------------------------------------------
- These public checkpoints are LIBERO/Franka-sim policies. Driving SO-101
  with them is out-of-distribution. Plan 14-03's bar is "a validated,
  non-held live action", not task success.
- SO-101 joint degrees are not LIBERO Franka EEF proprio. The client's wire
  contract (`base_http_adapter`) sends `joint_state`, which is accepted and
  ignored. The model gets the request's optional 8-float `proprio_state` if
  present, otherwise the checkpoint's own dataset-mean proprio as a neutral
  input.
- Only the FIRST action of each 8-step chunk is returned (receding horizon).
  The control loop re-queries every tick.

Colab launch: see vla_bridge/notebooks/http_openvla_oft.ipynb.
"""
import argparse
import base64
import os
import threading
import time
from pathlib import Path
from types import SimpleNamespace

import cv2
import numpy as np

DEFAULT_CHECKPOINT = "moojink/openvla-7b-oft-finetuned-libero-object"
# T-14-06: pin the reviewed HF revision (HF API sha read 2026-10-01), never `main`.
DEFAULT_REVISION = "4c89574e1c538b6c102f43f0526d60a9d3650148"
DEFAULT_LOCAL_DIR = "/content/openvla_oft_checkpoints"
ACTION_DIM = 7
PROPRIO_DIM = 8
MAX_IMAGE_SIDE = 1024
REQUIRED_IMAGES = ("overhead_left", "wrist")  # third-person full_image + wrist_image


def _download_pinned(checkpoint, revision, local_dir):
    """Materialise the exact pinned HF revision as a real directory.

    openvla-oft's local-checkpoint path (`get_vla` / `find_checkpoint_file`)
    then loads that directory. A repo-id load would always resolve `main`.
    """
    from huggingface_hub import snapshot_download

    target = Path(local_dir) / checkpoint.replace("/", "--") / revision
    return snapshot_download(repo_id=checkpoint, revision=revision, local_dir=str(target))


def _load_policy(cfg):
    """Load every OpenVLA-OFT component exactly as `vla-scripts/deploy.py` does.

    Needs the openvla-oft repo on `sys.path` and as the CWD (see `main()`).
    """
    from experiments.robot.openvla_utils import (
        get_action_head,
        get_processor,
        get_proprio_projector,
        get_vla,
        get_vla_action,
    )
    from prismatic.vla.constants import ACTION_DIM as REPO_ACTION_DIM
    from prismatic.vla.constants import PROPRIO_DIM as REPO_PROPRIO_DIM

    if (REPO_ACTION_DIM, REPO_PROPRIO_DIM) != (ACTION_DIM, PROPRIO_DIM):
        raise RuntimeError(
            f"openvla-oft constants resolved to ACTION_DIM={REPO_ACTION_DIM}, "
            f"PROPRIO_DIM={REPO_PROPRIO_DIM}; the LIBERO checkpoints need {ACTION_DIM}/{PROPRIO_DIM}"
        )
    vla = get_vla(cfg)
    proprio_projector = get_proprio_projector(cfg, vla.llm_dim, REPO_PROPRIO_DIM) if cfg.use_proprio else None
    action_head = get_action_head(cfg, vla.llm_dim)
    processor = get_processor(cfg)
    return dict(vla=vla, processor=processor, action_head=action_head,
                proprio_projector=proprio_projector, get_vla_action=get_vla_action)


class OpenVLAOFTModel:
    def __init__(self, checkpoint, revision=None, unnorm_key=None, local_dir=DEFAULT_LOCAL_DIR,
                 use_proprio=True, num_images_in_input=2, center_crop=True):
        pretrained = _download_pinned(checkpoint, revision, local_dir) if revision else checkpoint
        # The fields openvla_utils reads, with LIBERO GenerateConfig values.
        self.cfg = SimpleNamespace(
            model_family="openvla", pretrained_checkpoint=pretrained,
            use_l1_regression=True, use_diffusion=False,
            num_diffusion_steps_train=50, num_diffusion_steps_inference=50,
            use_film=False, num_images_in_input=num_images_in_input, use_proprio=use_proprio,
            center_crop=center_crop, num_open_loop_steps=8, lora_rank=32,
            unnorm_key=unnorm_key or "", load_in_8bit=False, load_in_4bit=False,
        )
        policy = _load_policy(self.cfg)
        self.vla = policy["vla"]
        self.processor = policy["processor"]
        self.action_head = policy["action_head"]
        self.proprio_projector = policy["proprio_projector"]
        self._get_vla_action = policy["get_vla_action"]

        keys = sorted(getattr(self.vla, "norm_stats", None) or {})
        if unnorm_key is None:
            if len(keys) != 1:
                raise ValueError(f"{checkpoint} has unnorm keys {keys}; pass --unnorm-key to choose one")
            unnorm_key = keys[0]
        elif unnorm_key not in keys:
            raise ValueError(f"unnorm key {unnorm_key!r} not in {checkpoint}'s dataset statistics {keys}")
        self.cfg.unnorm_key = unnorm_key
        self.neutral_proprio = np.asarray(
            self.vla.norm_stats[unnorm_key]["proprio"]["mean"], dtype=np.float64
        )
        self.model_version = f"{checkpoint}@{revision[:12] if revision else 'unpinned'}"
        self.metadata = dict(
            checkpoint=checkpoint, revision=revision or "unpinned", unnorm_key=unnorm_key,
            action_space="delta_eef_robosuite_osc_normalized", action_dim=ACTION_DIM,
            gripper="0_close_1_open", proprio_dim=PROPRIO_DIM, use_proprio=use_proprio,
            num_images_in_input=num_images_in_input, returns="first action of chunk",
        )

    def __call__(self, images, proprio_state, instruction):
        obs = {"full_image": images["overhead_left"], "wrist_image": images["wrist"]}
        if self.cfg.use_proprio:
            if proprio_state is None:
                state = self.neutral_proprio.copy()
            else:
                state = np.asarray(proprio_state, dtype=np.float64)
            if state.shape != (PROPRIO_DIM,) or not np.all(np.isfinite(state)):
                raise ValueError(f"proprio_state must be {PROPRIO_DIM} finite floats")
            obs["state"] = state
        actions = self._get_vla_action(
            self.cfg, self.vla, self.processor, obs, instruction,
            action_head=self.action_head, proprio_projector=self.proprio_projector, use_film=False,
        )
        first = np.asarray(actions[0], dtype=np.float64).reshape(-1)
        if first.shape != (ACTION_DIM,):
            raise RuntimeError(f"model returned action of shape {first.shape}, expected ({ACTION_DIM},)")
        return [float(v) for v in first]


def _decode_rgb(b64_png):
    """base64 PNG (OpenCV BGR-encoded by the client) -> HxWx3 uint8 RGB."""
    if not isinstance(b64_png, str):
        raise TypeError("image must be a base64 string")
    raw = base64.b64decode(b64_png, validate=True)
    frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError("invalid PNG")
    if max(frame.shape[:2]) > MAX_IMAGE_SIDE:
        raise ValueError(f"image dimensions must be <= {MAX_IMAGE_SIDE}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _parse_proprio(value):
    if value is None:
        return None
    if not isinstance(value, list) or len(value) != PROPRIO_DIM:
        raise ValueError(f"proprio_state must be a list of {PROPRIO_DIM} numbers")
    state = [float(v) for v in value]
    if not all(np.isfinite(state)):
        raise ValueError("proprio_state must be finite")
    return state


def create_app(infer):
    from flask import Flask, jsonify, request
    app = Flask(__name__)
    app.config['MAX_CONTENT_LENGTH'] = 20 * 1024 * 1024
    inference_lock = threading.Lock()

    @app.get('/health')
    def health():
        return jsonify(status='ready', protocol_version=1, model=infer.metadata)

    @app.post('/action')
    def action():
        if not inference_lock.acquire(blocking=False):
            return jsonify(error='openvla-oft service busy'), 429
        try:
            body = request.get_json(silent=True)
            if not isinstance(body, dict):
                raise ValueError('expected a JSON object body')
            images_in = body['images']
            if not isinstance(images_in, dict):
                raise TypeError('images must be an object')
            images = {key: _decode_rgb(images_in[key]) for key in REQUIRED_IMAGES}
            instruction = body['instruction']
            if not isinstance(instruction, str) or not instruction.strip():
                raise ValueError('instruction must be a non-empty string')
            proprio_state = _parse_proprio(body.get('proprio_state'))
            joint_state = body.get('joint_state')
            if joint_state is not None and not isinstance(joint_state, dict):
                raise TypeError('joint_state must be an object')
            started = time.perf_counter()
            result = infer(images, proprio_state, instruction)
            elapsed = (time.perf_counter() - started) * 1000
            return jsonify(action=[float(v) for v in result], model_version=infer.model_version,
                           inference_ms=elapsed)
        except (ValueError, KeyError, TypeError) as exc:
            return jsonify(error=str(exc)), 400
        finally:
            inference_lock.release()
    return app


def main():
    import sys
    parser = argparse.ArgumentParser()
    parser.add_argument('--repo', required=True, help='openvla-oft checkout (github.com/moojink/openvla-oft)')
    parser.add_argument('--checkpoint', default=DEFAULT_CHECKPOINT)
    parser.add_argument('--revision', default=DEFAULT_REVISION,
                        help="pinned HF commit sha; pass '' only to deliberately load the unpinned repo id")
    parser.add_argument('--unnorm-key', default=None)
    parser.add_argument('--local-dir', default=DEFAULT_LOCAL_DIR)
    parser.add_argument('--port', type=int, default=3001)
    args = parser.parse_args()
    repo = str(Path(args.repo).resolve())
    sys.path.insert(0, repo)
    # openvla_utils.check_model_logic_mismatch() walks ./prismatic/ relative to the CWD.
    os.chdir(repo)
    model = OpenVLAOFTModel(args.checkpoint, revision=args.revision or None,
                            unnorm_key=args.unnorm_key, local_dir=args.local_dir)
    # CUDA kernels / lazy init happen before the socket opens and readiness is reported.
    warmup = np.zeros((224, 224, 3), dtype=np.uint8)
    model({'overhead_left': warmup, 'wrist': warmup}, None, 'warm up')
    print('OpenVLA-OFT model loaded and warmed; ready', flush=True)
    create_app(model).run(host='0.0.0.0', port=args.port, threaded=True, use_reloader=False)


if __name__ == '__main__':
    main()


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '59f3fd30a1e38cb02bbf131df4fba7c2c534acbf0ed0525ad834b8dbf12fc8c4'
actual = hashlib.sha256(Path('/content/openvla_oft_server.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/openvla_oft_server.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/openvla_oft_server.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/openvla_oft_server.py matches control/vla_bridge/openvla_oft_server.py (sha256 {actual[:12]})')

## Launch

The launch cell first stops any earlier server: every process whose command line contains one of `openvla_oft_server.py`, and whatever listens on port 3001.
It then starts this notebook's server as a separate OS process (unbuffered, log at `/content/openvla_oft_server.log`).
It only succeeds once **its own** still-running process owns port 3001.
A stale server answering on the port is an error, never a successful launch (in the 2026-10-18 live session a stale stock server made a diffusion launch look successful).

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import start_server, stop_servers, wait_until_ready

PORT = 3001
OFT_LOG = '/content/openvla_oft_server.log'
STOP_PATTERNS = ['openvla_oft_server.py']
OFT_DIR = '/content/openvla-oft'
OFT_PYTHON = '/content/openvla_oft_venv/bin/python'

stop_servers(PORT, STOP_PATTERNS)
oft_server_process = start_server(
    [OFT_PYTHON, '-u', '/content/openvla_oft_server.py', '--repo', OFT_DIR, '--port', '3001'],
    OFT_LOG,
    env_overrides={'KMP_DUPLICATE_LIB_OK': 'TRUE', 'OMP_NUM_THREADS': '1'},
)
health = wait_until_ready(
    oft_server_process, PORT, OFT_LOG,
    timeout_s=1800, health_url='http://127.0.0.1:3001/health',
)
print('OpenVLA-OFT adapter ready')

## Open an HTTP tunnel to port 3001

In [ ]:
from pyngrok import ngrok

oft_tunnel = ngrok.connect(3001, "http")  # plain JSON-over-HTTP
print(f"OpenVLA-OFT endpoint: {oft_tunnel.public_url}")
SMOKE = '.venv/bin/python -m vla_bridge.backend_smoke --backend openvla-oft --server-address https://<oft_tunnel>'
EPISODE = '.venv/bin/python run_vla_episode.py --backend openvla-oft --server-address https://OFT_TUNNEL.ngrok-free.app --out outputs/openvla_oft_UNIQUE_NAME --max-steps 20'
for placeholder in ('https://<oft_tunnel>', 'https://OFT_TUNNEL.ngrok-free.app'):
    SMOKE = SMOKE.replace(placeholder, oft_tunnel.public_url)
    EPISODE = EPISODE.replace(placeholder, oft_tunnel.public_url)
print('Local smoke (run in control/):')
print(SMOKE)
print('Local episode (run in control/):')
print(EPISODE)

## Run locally

Then, locally in `control/` (pass the base URL; the client appends `/action`):

```bash
.venv/bin/python run_vla_episode.py --backend openvla-oft \
  --server-address https://OFT_TUNNEL.ngrok-free.app \
  --out outputs/openvla_oft_UNIQUE_NAME --max-steps 20
```

Smoke run first (synthetic hardware, the arm cannot move; the tunnel cell prints it with the URL filled in):

```bash
.venv/bin/python -m vla_bridge.backend_smoke --backend openvla-oft --server-address https://<oft_tunnel>
```

## Diagnostics

Run any of these after the launch cell, also from a restarted kernel: each cell re-declares
what it needs.
- Log: the tail of `/content/openvla_oft_server.log`, then every line matching `ERROR`, `Error`, `Traceback`,
  `Exception`, `OutOfMemory`, `out of memory`, `Killed` or `beginAllocateToPool`.
- Processes: every known server process on this runtime (stale servers from other notebooks
  show up here); fails unless `openvla_oft_server.py` owns port 3001.
- Package versions, GPU memory, and `/health`.

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import log_report

log_report('/content/openvla_oft_server.log')

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import process_report

process_report(3001, 'openvla_oft_server.py', globals().get('oft_server_process'))

In [ ]:
import subprocess

OFT_PYTHON = '/content/openvla_oft_venv/bin/python'
probe = (
    "import importlib.metadata as m\n"
    "for p in ('torch', 'transformers', 'tensorflow', 'flask'):\n"
    "    try:\n"
    "        print(p, m.version(p))\n"
    "    except m.PackageNotFoundError:\n"
    "        print(p, 'not installed')\n"
)
result = subprocess.run([OFT_PYTHON, '-c', probe], capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-2000:])

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import gpu_report

gpu_report()

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import health_report

health_report(3001)

## Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted server is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day: the cell below disconnects this notebook's tunnel, stops the
ngrok agent, stops the server and confirms port 3001 is free. Run it only
after the local episode or smoke run has finished.

In [ ]:
# OpenVLA-OFT teardown -- every session, after the local run has finished.
import sys

sys.path.insert(0, '/content')
from colab_server_utils import stop_server
from pyngrok import ngrok

if 'oft_tunnel' in globals():
    try:
        ngrok.disconnect(oft_tunnel.public_url)
        print(f'Disconnected {oft_tunnel.public_url}')
    except Exception as exc:
        print(f'Tunnel already closed ({type(exc).__name__}: {exc})')
ngrok.kill()
print('ngrok agent stopped.')
stop_server(globals().get('oft_server_process'), 3001, ['openvla_oft_server.py'])